In [20]:
# ===============================
# IMPORTS
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

import spacy
from sklearn.base import BaseEstimator, TransformerMixin

# Load spaCy model
nlp = spacy.load("en_core_web_sm")


# ===============================
# LOAD DATA
# ===============================
def load_data(filepath):
    df = pd.read_csv(filepath)
    return df


def explore_data(df):
    print("Shape:", df.shape)
    print("\nMissing Values:\n", df.isnull().sum())
    print("\nTarget Distribution:\n", df["Recommended IND"].value_counts(normalize=True))


# ===============================
# SPACY TRANSFORMER (INSIDE PIPELINE)
# ===============================
class AddSpacyFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        texts = X["Review Text"].fillna("").astype(str).tolist()

        features = []
        for doc in nlp.pipe(texts, disable=["parser", "ner"]):
            features.append([
                len(doc),
                sum(1 for t in doc if t.pos_ == "NOUN"),
                sum(1 for t in doc if t.pos_ == "VERB"),
                sum(1 for t in doc if t.is_stop)
            ])

        features = np.array(features)

        X["doc_length"] = features[:, 0]
        X["noun_count"] = features[:, 1]
        X["verb_count"] = features[:, 2]
        X["stopword_count"] = features[:, 3]

        return X


# ===============================
# PIPELINE BUILDER (NO SPACY)
# ===============================
def build_pipeline(text_col, num_cols, cat_cols):

    text_transformer = Pipeline([
        ('vect', CountVectorizer(stop_words='english')),
        ('tfidf', TfidfTransformer())
    ])

    num_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('text', text_transformer, text_col),
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000))
    ])

    return pipeline


# ===============================
# TRAIN
# ===============================
def train_model(pipeline, X_train, y_train):
    pipeline.fit(X_train.copy(), y_train.copy())
    return pipeline


# ===============================
# EVALUATE
# ===============================
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test.copy())
    print("\nModel Evaluation:\n")
    print(classification_report(y_test, y_pred))


# ===============================
# TUNE MODEL (NO SPACY)
# ===============================
def tune_model(pipeline, X_train, y_train):

    param_grid = {
        "model__C": [0.1, 1, 10],
        "model__solver": ["lbfgs"]
    }

    grid = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        cv=3,
        n_jobs=-1
    )

    grid.fit(X_train.copy(), y_train.copy())

    print("\nBest Parameters:", grid.best_params_)

    return grid.best_estimator_


# ===============================
# MAIN FLOW
# ===============================

# Load data
df = load_data("starter/data/reviews.csv")

# Explore
explore_data(df)

# Columns
text_col = "Review Text"
num_cols = ["Age", "doc_length", "noun_count", "verb_count", "stopword_count"]
cat_cols = ["Department Name", "Class Name"]
target = "Recommended IND"

# Clean
df = df.dropna(subset=[text_col, target])

X = df.drop(target, axis=1)
y = df[target]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------
# 1. Build tuning pipeline (NO spaCy)
# -------------------------------
tuning_pipeline = build_pipeline(text_col, ["Age"], cat_cols)

# -------------------------------
# 2. Train & evaluate
# -------------------------------
model = train_model(tuning_pipeline, X_train, y_train)
evaluate_model(model, X_test, y_test)

# -------------------------------
# 3. Tune model
# -------------------------------
best_model = tune_model(tuning_pipeline, X_train, y_train)

# -------------------------------
# 4. Build FINAL pipeline (WITH spaCy)
# -------------------------------
final_pipeline = Pipeline([
    ('spacy_features', AddSpacyFeatures()),
    ('preprocessor', build_pipeline(text_col, num_cols, cat_cols).named_steps['preprocessor']),
    ('model', LogisticRegression(max_iter=1000))
])

# Apply tuned params
final_pipeline.set_params(**best_model.get_params())

# -------------------------------
# 5. Train final pipeline
# -------------------------------
final_pipeline.fit(X_train, y_train)

# -------------------------------
# 6. Final evaluation
# -------------------------------
print("\nFinal Model Performance:")
evaluate_model(final_pipeline, X_test, y_test)


Shape: (18442, 9)

Missing Values:
 Clothing ID                0
Age                        0
Title                      0
Review Text                0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
Recommended IND            0
dtype: int64

Target Distribution:
 Recommended IND
1    0.816235
0    0.183765
Name: proportion, dtype: float64

Model Evaluation:

              precision    recall  f1-score   support

           0       0.77      0.52      0.62       692
           1       0.90      0.96      0.93      2997

    accuracy                           0.88      3689
   macro avg       0.83      0.74      0.77      3689
weighted avg       0.87      0.88      0.87      3689


Best Parameters: {'model__C': 10, 'model__solver': 'lbfgs'}

Final Model Performance:

Model Evaluation:

              precision    recall  f1-score   support

           0       0.71      0.61      0.66       692
           1       0.91    

In [21]:
import joblib

#save the model
best_model = tune_model(final_pipeline, X_train, y_train)

#load the model for testing
joblib.dump(best_model, "model.pkl")

#test prediction
loaded_model = joblib.load("model.pkl")
sample_pred = loaded_model.predict(X_test[:5])
print(sample_pred)



Best Parameters: {'model__C': 10, 'model__solver': 'lbfgs'}
[1 1 1 1 0]
